# W13A · 첫 추천을 내 쇼핑몰에 띄우기

딥러닝응용I(추천시스템) · 2026년 11월 24일 · 01분반

먼저 기본 화면을 열고 상품을 담습니다. 작은 주문표에서 계산해 본 뒤 두 함수를 직접 작성합니다.
수업은 생성형 AI 없이 진행합니다. 과제 1은 **12월 4일(금) 23:59**까지입니다.

[학생용 슬라이드](https://github.com/lunalab-ai/recommender/releases/download/2026-fall-w13a-r1/w13a-first-recommendation.pdf)
· [과제 1 안내](https://github.com/lunalab-ai/recommender/blob/2026-fall-w13a-r1/assignments/student/w13a-assignment1.md)

## 1. 기본 화면 열기

전체 실행을 먼저 누르면 수업용 데이터와 시연 화면을 현재 런타임에 준비합니다. 무료 CPU에서 실행합니다.
아래 출력에서 **기본 쇼핑몰**을 받으세요. 파일을 브라우저로 열고 **3단 케이크 스탠드**를 담은 뒤
추천 방식을 **인기순 → 아이템 CF · 3**으로 바꿉니다. 실험 통계의 표는 이 단계에서 18건의 시연 주문으로 표시됩니다.


In [ ]:
import base64
import inspect
import io
import json
import os
import sys
import zipfile
from math import sqrt
from pathlib import Path

from IPython.display import FileLink, display

root = Path("w13a-project").resolve()
root.mkdir(exist_ok=True)
bundle_text = (
    '...'
)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(bundle_text))) as bundle:
    for asset in ("retail.json", "w13a_basket.py", "shop.template.html", "media.json", "demo-payload.json"):
        (root / asset).write_bytes(bundle.read(asset))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import w13a_basket as lab

data = lab.read_retail(root / "retail.json")
template = (root / "shop.template.html").read_text(encoding="utf-8")
media = json.loads((root / "media.json").read_text(encoding="utf-8"))
demo = json.loads((root / "demo-payload.json").read_text(encoding="utf-8"))
demo["media"] = media
(root / "w13a-default-shop.html").write_text(lab.render_shop(template, demo), encoding="utf-8", newline="\n")
print("W13A_READY", len(data["train"]), len(data["catalog"]), len(data["validation"]))
print("학습 주문 · 상품 수 · 검증 주문 순서입니다.")
display(FileLink(os.path.relpath(root / "w13a-default-shop.html")))


Colab에서는 아래 셀의 파일 이름을 그대로 두고 **DOWNLOAD = True**로 바꾼 뒤 그 셀만 실행해도 됩니다.


In [ ]:
DOWNLOAD = False
if DOWNLOAD:
    from google.colab import files
    files.download(str(root / "w13a-default-shop.html"))


## 2. 복습 · 주문을 열 벡터로 읽기

영화 평점표에서 코사인의 분자와 두 노름을 계산한 과정을 다시 떠올리세요.
이번에는 행이 **주문**, 열이 **상품**입니다. 1은 구매가 기록됐음을 뜻합니다.
기존 자료에서 [W03B 2절](https://github.com/lunalab-ai/recommender/blob/2026-fall-w03b/course/notion/sessions/w03b-cf-step-by-step.md)과
[W04B Colab E1·E3](https://colab.research.google.com/github/lunalab-ai/recommender/blob/2026-fall-w04b/notebooks/student/w04b-cf-synthesis.ipynb)을 복습할 수 있습니다.

| 주문 | 머그 | 티스푼 | 조명 | 파우치 | 노트 |
|---|---:|---:|---:|---:|---:|
| 1 | 1 | 1 | 0 | 0 | 0 |
| 2 | 1 | 1 | 1 | 0 | 0 |
| 3 | 1 | 0 | 0 | 1 | 0 |
| 4 | 0 | 0 | 1 | 1 | 0 |
| 5 | 0 | 0 | 1 | 0 | 1 |
| 6 | 0 | 0 | 1 | 0 | 1 |

머그와 티스푼의 공통 주문은 2건입니다. 두 상품의 전체 주문 수는 3건과 2건이므로 코사인은 $2 / \sqrt{3 \times 2}$입니다.
머그와 조명은 공통 주문 1건, 각각 3건과 4건입니다. 기준을 3으로 올리면 머그의 두 연결이 모두 빠집니다.


In [ ]:
toy = lab.order_matrix([[0,1],[0,1,2],[0,3],[2,3],[2,4],[2,4]], 5)
toy_shared = toy.T @ toy
assert toy.shape == (6, 5) and toy_shared[0, 1] == 2 and toy_shared[0, 2] == 1
assert (len(data["train"]), len(data["catalog"]), len(data["validation"])) == (4046, 250, 674)
names = ("머그","티스푼","조명","파우치","노트")
print("전체 주문수", dict(zip(names, toy.sum(axis=0).tolist())))
for j in (1, 2):
    co = int(toy_shared[0, j])
    left, right = int(toy_shared[0, 0]), int(toy_shared[j, j])
    print(names[j], "공통", co, "분모", round(sqrt(left * right), 3), "유사도", round(co/sqrt(left*right),3))


## 3. 따라 실행 · 인기순

각 열을 세로로 더해 상품별 주문 횟수를 얻습니다. 아래를 실행하면 합성 주문에서 **조명**이 첫 번째입니다.
담은 물건이 있을 때에는 같은 계산으로 후보를 정렬한 뒤 장바구니 상품을 제외합니다.


In [ ]:
pop = toy.sum(axis=0)
print([names[j] for j in sorted(range(len(names)), key=lambda j: (-int(pop[j]), j))])


## 4. 내 코드 두 곳

**연습 1**. 두 상품의 공통 주문 수와 각각의 전체 주문 수가 함수로 들어갑니다. 분모가 0이면 0을 반환합니다.
반환식에서 기본 제공되는 `sqrt`를 사용하세요. 검사와 재채점을 위해 두 함수는 각각 `return` 식 한 줄로 작성합니다.


In [ ]:
def item_cosine(co, left, right):
    return None  # 교체: 두 개의 전체 주문 수로 분모를 계산하세요.


**연습 2**. 후보 상품 번호 하나와 현재 장바구니의 상품 번호들이 들어옵니다.
장바구니에 없는 상품을 남기는 참/거짓의 식으로 바꿔 보세요.


In [ ]:
def is_candidate(item, basket):
    return None  # 교체: 후보가 장바구니에 들어있는지 확인하세요.


## 5. 짧은 점검을 통과하고 비교표 만들기

공개 점검의 두 항목에 통과하면 다음 실험을 실행하세요.
`False`가 나오면 각각 분모의 0 처리와 장바구니에 이미 담긴 두 상품을 확인하세요.
상품 하나를 가리고 남은 상품만 입력해 같은 검증 주문에서 인기순과 CF 1·3·5를 비교합니다.
Hit@5는 가린 상품을 맞힌 비율입니다. NDCG@5는 맞혔을 때 더 앞에 올린 결과에 더 큰 값을 줍니다.
다시 볼 자료는 [W05A 5~6절](https://github.com/lunalab-ai/recommender/blob/2026-fall-w05a-v2/course/notion/sessions/w05a-model-comparison.md)입니다.


In [ ]:
import pandas as pd

completed = item_cosine(1, 1, 1) is not None and is_candidate(0, (1,)) is not None
if completed:
    assert abs(item_cosine(2, 3, 2) - 0.8164965809277261) < 1e-9 and item_cosine(0, 0, 2) == 0
    assert is_candidate(2, (0, 1)) and not is_candidate(1, (0, 1))
    print("W13A_TWO_EXERCISES_PASS")
    model = lab.fit(data, item_cosine)
    report = lab.evaluate(data, model, is_candidate)
    display(pd.DataFrame(report))
else:
    print("기본 시연은 열 수 있습니다. 내 실험표를 만들려면 연습 1과 2의 반환식을 모두 바꾸고 다시 실행하세요.")


## 6. 내 쇼핑몰에서 적용하고 제출하기

실행하면 내 점수를 내장한 `shop.html`이 생깁니다. 처음 웹에는 인기순이 선택됩니다.
**실험 통계**로 이동해 비교표를 읽고 **Hit@5 → NDCG@5** 순으로 동점을 판단한 뒤 출시 행을 선택합니다.
다시 상품 화면에서 상품을 담아 적용 결과를 확인하고, 통계 페이지의 **제출 ZIP 받기**로 파일을 저장합니다.
다시 열어도 이 장바구니 계산은 브라우저에서 실행됩니다. Colab은 주문표·점수·성능표를 준비합니다.


In [ ]:
if completed:
    source = inspect.getsource(item_cosine).strip()+"\n\n"+inspect.getsource(is_candidate).strip()+"\n"
    payload = lab.browser_payload(data, model, report, "popularity", source, media)
    (root / "shop.html").write_text(lab.render_shop(template, payload), encoding="utf-8")
    print("W13A_STUDENT_SITE_READY")
    display(FileLink(os.path.relpath(root / "shop.html")))


In [ ]:
DOWNLOAD_MY_SITE = False
if DOWNLOAD_MY_SITE and completed:
    from google.colab import files
    files.download(str(root / "shop.html"))


### 과제 1 제출 전 점검

1. 통계 페이지에 인기순과 CF 1·3·5 네 행이 표시되는지 확인합니다.
2. 출시 모델을 직접 선택하고, 장바구니로 돌아가 해당 모델의 추천이 나오는지 확인합니다.
3. 실험 통계에서 받은 `w13a-assignment1.zip` 하나를 SmartClass의 해당 **과제**에 제출합니다.
   압축 안에는 `shop.html`, `submission.json`, `metrics.csv`, `student_model.py`가 있어야 합니다.
4. **12월 4일(금) 23:59**가 마감입니다. 표에는 설명문을 추가할 필요가 없습니다.

거래 자료 Chen (2015), UCI Online Retail, [CC BY 4.0과 출처](https://archive.ics.uci.edu/dataset/352/online%2Bretail).
취소·음수 거래를 정리한 뒤 학습 기간에서 250개 상품을 선정했습니다. 분류와 사진은 수업용 예시입니다.
